![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, LangChain and Milvus to create and deploy RAG function

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.

## Notebook content

This notebook contains the steps and code to demonstrate support of creating and deploying Retrieval Augmented Generation in watsonx.ai. It introduces commands for data retrieval, knowledge base building & querying, model testing, deploying a RAG solution for general use.

Some familiarity with Python is helpful. This notebook uses Python 3.11.

#### About Retrieval Augmented Generation
Retrieval Augmented Generation (RAG) is a versatile pattern that can unlock a number of use cases requiring factual recall of information, such as querying a knowledge base in natural language.

In its simplest form, RAG requires 3 steps:

- Index knowledge base passages (once)
- Retrieve relevant passage(s) from knowledge base (for every user query)
- Generate a response by feeding retrieved passage into a large language model (for every user query)

## Contents

This notebook contains the following parts:

- [Setup](#setup)
- [Data (test) loading](#data)
- [Set up connectivity information to Milvus](#milvus_conn)
- [Set up VectorStore with Milvus credentials](#vectorstore)
- [Create and deploy RAG solution](#deploy)
- [Calculate rougeL metric](#evaluate)

<a id="setup"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).

### Install and import dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install rouge-score | tail -n 1
%pip install -U "ibm_watsonx_ai>=1.4.0" | tail -n 1
%pip install -U "langchain>=0.3,<1.0" | tail -n 1
%pip install -U "langchain-milvus>=0.1,<0.2" | tail -n 1

In [2]:
import getpass
import os

import wget
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import Embeddings
from ibm_watsonx_ai.foundation_models.extensions.rag import VectorStore
from ibm_watsonx_ai.foundation_models.extensions.rag.utils import verbose_search
from ibm_watsonx_ai.foundation_models.prompts import (
    PromptTemplate,
    PromptTemplateManager,
)
from ibm_watsonx_ai.helpers import DataConnection
from IPython.display import Markdown, display
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rouge_score import rouge_scorer

### Define the watsonx.ai credentials
Use the code cell below to define the watsonx.ai credentials that are required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="noopener no referrer">Managing user API keys</a>.

In [3]:
credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your api key and hit enter: "),
)

### Defining the project id
The Foundation Model requires project id that provides the context for the call. We will obtain the id from the project in which this notebook runs. Otherwise, please provide the project id.


In [4]:
try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id and hit enter: ")

### Defining the space id
Deployed functions are available on deployment spaces. RAG we will create, will be a deployed function. You need to provide space id.

In [5]:
space_id = input("Please enter your space_id and hit enter: ")

### Initialize client
Create an instance of `APIClient` and set the default project.

In [6]:
client = APIClient(credentials, project_id=project_id)

### Defining the prompt id

We will use PromptTemplate to create a template for our RAG LLM query. If you don't have the PromptTemplate created in your project, this code will create an example one.

In [7]:
prompt_id = (
    input(
        "Please enter your prompt template asset id and hit enter, if not provided, a new one would be created: "
    )
    or None
)

if prompt_id is None:
    PROMPT_INSTRUCTION = """
    Use the following pieces of documents to answer the question
    at the end. If you don't know the answer, just say that you
    don't know, don't try to make up an answer. Use three sentences
    maximum. Keep the answer as concise as possible. do not include
    question in your response.Your answers should not include any
    harmful, unethical, racist, sexist, toxic, dangerous, or illegal
    content. Please ensure that your responses are socially unbiased
    and positive in nature.\nPlease provide a concise professional
    response.
    """
    prompt_mgr = PromptTemplateManager(credentials=credentials, project_id=project_id)
    prompt_template = PromptTemplate(
        name="RAG_prompt_template",
        model_id=client.foundation_models.TextModels.LLAMA_3_3_70B_INSTRUCT,
        input_variables=["question", "reference_documents"],
        instruction=PROMPT_INSTRUCTION,
        input_text="{reference_documents}\nQuestion:{question}\nAnswer:",
    )
    stored_prompt_template = prompt_mgr.store_prompt(prompt_template=prompt_template)
    prompt_id = stored_prompt_template.prompt_id

### Build up knowledge base

The current state-of-the-art in RAG is to create dense vector representations of the knowledge base in order to calculate the semantic similarity to a given user query.

We can generate dense vector representations using embedding models. In this notebook, we use IBM's <a href="https://www.ibm.com/products/watsonx-ai/foundation-models#Embedding+model+library">IBM_SLATE_30M_ENG</a> model to embed both the knowledge base passages and user queries.

A vector database is optimized for dense vector indexing and retrieval. This notebook uses <a href="https://python.langchain.com/docs/integrations/vectorstores/Milvus#basic-example" target="_blank" rel="noopener no referrer">Milvus</a>, an open-source vector database.

The dataset we are using is already split into self-contained passages that can be ingested by Milvus. 

The size of each passage is limited by the embedding model's context window (which is 512 tokens for `IBM Slate 30M`).

### Load knowledge base documents

Load set of documents used further to build knowledge base and store them as a project asset.

In [8]:
filename = "psgs.tsv"
url = f"https://raw.github.com/IBM/watsonx-ai-samples/master/cloud/data/RAG/{filename}"
if not os.path.isfile(filename):
    wget.download(url)

asset_details = client.data_assets.create(name=filename, file_path=filename)

Creating data asset...
SUCCESS


### Read and prepare documents
Read documents using `DataConnection` and prepare them for vector database ingestion by combining title and text.

In [9]:
data_connection = DataConnection(data_asset_id=client.data_assets.get_id(asset_details))
data_connection.set_client(client)
documents = data_connection.read(csv_separator="\t")

In [10]:
documents["indextext"] = documents["title"].astype(str) + "\n" + documents["text"]
documents = documents[:1000]
documents.head()

,id,text,title,indextext
0,1.0,History of Idaho - wikipedia History of Idaho ...,History of Idaho,History of Idaho\nHistory of Idaho - wikipedia...
1,2.0,"1957 . Location Cataldo , Idaho Built 1848 Arc...",History of Idaho,"History of Idaho\n1957 . Location Cataldo , Id..."
2,3.0,"of the Columbia was created in June 1816 , and...",History of Idaho,History of Idaho\nof the Columbia was created ...
3,4.0,"Canyon , he concluded that water transport was...",History of Idaho,"History of Idaho\nCanyon , he concluded that w..."
4,5.0,"1842 , Father Pierre - Jean De Smet , with Fr....",History of Idaho,"History of Idaho\n1842 , Father Pierre - Jean ..."


### Create an embedding function for VectorStore

Note that you can feed a custom embedding function to be used by Milvus. The performance of Milvus may differ depending on the embedding model used. 

In [11]:
embeddings = Embeddings(
    model_id=client.foundation_models.EmbeddingModels.SLATE_30M_ENGLISH_RTRVR_V2,
    credentials=credentials,
    project_id=project_id,
)

<a id="elastic_conn"></a>
## Set up connectivity information to Milvus

**This notebook focuses on self-managed Milvus cluster using <a href="https://cloud.ibm.com/docs/watsonxdata?topic=watsonxdata-adding-milvus-service" target="_blank" rel="noopener no referrer">IBM watsonx.data.</a>**

The following cell retrieves the Milvus username, password, host and port from the environment if available and prompts you otherwise.

You can provide a connection asset ID to read all required connection data from it. Before doing so, make sure that connection asset was created in your project.

In [12]:
connection_id = (
    input(
        "Provide connection asset ID in your project. Skip this, if you wish to type credentials by hand and hit enter: "
    )
    or None
)

if connection_id is None:
    try:
        username = os.environ["USERNAME"]
    except KeyError:
        username = input("Please enter your Milvus user name and hit enter: ")
    try:
        password = os.environ["PASSWORD"]
    except KeyError:
        password = getpass.getpass("Please enter your Milvus password and hit enter: ")
    try:
        host = os.environ["HOST"]
    except KeyError:
        host = input("Please enter your Milvus hostname and hit enter: ")
    try:
        port = os.environ["PORT"]
    except KeyError:
        port = input("Please enter your Milvus port number and hit enter: ")
    try:
        ssl = os.environ["SSL"]
    except:
        ssl = bool(
            input(
                "Please enter ('y'/anything) if your Milvus instance has SSL enabled. Skip if it is not: "
            )
        )

    # Create connection
    milvus_data_source_type_id = client.connections.get_datasource_type_uid_by_name(
        "milvus"
    )
    details = client.connections.create(
        {
            client.connections.ConfigurationMetaNames.NAME: "Milvus Connection",
            client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection created by the sample notebook",
            client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: milvus_data_source_type_id,
            client.connections.ConfigurationMetaNames.PROPERTIES: {
                "host": host,
                "port": port,
                "username": username,
                "password": password,
                "ssl": ssl,
            },
        }
    )

    connection_id = client.connections.get_id(details)

Creating connections...
SUCCESS


<a id="vectorstore"></a>
## Set up VectorStore with Milvus credentials 

Create a VectorStore class that automatically detects the database type (in our case it will be Milvus) and allows us to add, search and delete documents.

It works as a wrapper for LangChain VectorStore classes. You can customize the settings as long as it is supported. Consult the LangChain documentation for more information about <a href="https://api.python.langchain.com/en/latest/vectorstores/langchain_community.vectorstores.milvus.Milvus.html" target="_blank" rel="noopener no referrer">Milvus</a> connector.

Provide the name of your Milvus index for subsequent operations:

In [13]:
index_name = input("Please enter Milvus index name and hit enter: ")

In [14]:
vector_store = VectorStore(
    client=client,
    embeddings=embeddings,
    connection_id=connection_id,
    index_name=index_name,
    secure=True,
)

In [15]:
client.set.default_space(space_id)

Unsetting the project_id ...


'SUCCESS'

<a id="milvus_index"></a>
### Embed and index documents with Milvus

**Note: Could take several minutes if you don't have pre-built indices**

In [16]:
texts = documents.indextext.tolist()
metadatas = [
    {"title": title, "id": doc_id}
    for (title, doc_id) in zip(documents.title, documents.id)
]
docs_to_add = [
    Document(page_content=text, metadata=metadata)
    for text, metadata in zip(texts, metadatas)
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=10)
docs_to_add_split = text_splitter.split_documents(docs_to_add)

ids = vector_store.add_documents(docs_to_add_split, batch_size=200)

Verify the number of documents loaded into the Milvus.

In [17]:
doc_count = vector_store.count()
doc_count

4051

Let's search for an example document as a sample. Note the embedding in the vector field, that was generated with the sentence transformer.

In [18]:
vector_store.search("United States of America", k=5, verbose=True)

**Question:** United States of America

,page_content,id,pk,title
0,", D.C. States / Territories Alabama Alaska Ame...",639.0,9a4eb29daf1b4658f073e569f69ae15fc2959b3e4d2469...,"United States Senate elections, 2018"
1,"United States , 1797 -- 1801 1st Vice Presiden...",918.0,7d3d14905ddb5242dfeb01b9bfa682e9b3f3928636b7ec...,Founding Fathers of the United States
2,"the United States of America , which was recog...",521.0,380b686d2cb5b91bac2768e97db84ee75a9b678d720a4b...,British colonization of the Americas
3,1937 1938 1939 1940 1941 1942 1943 1944 1945 1...,22.0,3eaa26033406bb82f881cf6a75390b0c8248ebadce2853...,History of Idaho
4,of Congress further identifies the Articles of...,893.0,3244e754ea5f3a09edc6ef3eb30e22c20789c3ae796d11...,Founding Fathers of the United States


[Document(metadata={'title': 'United States Senate elections, 2018', 'id': 639.0, 'pk': '9a4eb29daf1b4658f073e569f69ae15fc2959b3e4d24694e292af9c9be742b48'}, page_content=', D.C. States / Territories Alabama Alaska American Samoa Arizona Arkansas California Colorado Connecticut Delaware Florida Georgia Guam Hawaii Idaho Illinois Indiana Iowa Kansas Kentucky Louisiana Maine Maryland Massachusetts Michigan Minnesota Mississippi Missouri Montana Nebraska Nevada New Hampshire New Jersey New Mexico New York North Carolina North Dakota Ohio Oklahoma Oregon Pennsylvania Puerto Rico'),
 Document(metadata={'title': 'Founding Fathers of the United States', 'id': 918.0, 'pk': '7d3d14905ddb5242dfeb01b9bfa682e9b3f3928636b7eca129ef02910309c06c'}, page_content='United States , 1797 -- 1801 1st Vice President of the United States , 1789 -- 1797 U.S. Ambassador to the United Kingdom , 1785 -- 1788 U.S. Ambassador to the Netherlands , 1782 -- 1788 Delegate , Second Continental Congress , 1775 -- 1778 Del

<a id="deploy"></a>
## Create and deploy RAG solution

### Define ai service code

Deployed function for RAG should implement the functionality of retrieval and augmenting the prompt for the LLM model.
Function defined can be used as an example. To modify the deployed function behaviour, change the values in `custom`.

In [19]:
custom = {
    "url": credentials.url,
    "space_id": space_id,
    "retriever": {"method": "simple", "number_of_chunks": 5},
    "vector_store": vector_store.to_dict(),
    "prompt_template_text": "\n    Use the following pieces of documents to answer the question\n    at the end. If you don't know the answer, just say that you\n    don't know, don't try to make up an answer. Use three sentences\n    maximum. Keep the answer as concise as possible. do not include\n    question in your response.Your answers should not include any\n    harmful, unethical, racist, sexist, toxic, dangerous, or illegal\n    content. Please ensure that your responses are socially unbiased\n    and positive in nature.\nPlease provide a concise professional\n    response.\n    \n\n{reference_documents}\nQuestion:{question}\nAnswer:",
    "context_template_text": None,
    "model": {
        "model_id": "meta-llama/llama-3-2-11b-vision-instruct",
        "params": {
            "decoding_method": "greedy",
            "min_new_tokens": 1,
            "max_new_tokens": 200,
        },
        "project_id": None,
        "space_id": space_id,
    },
    "inference_function_params": {},
}


def deployable_ai_service(context, **custom):
    """
    Deployed function.

    Input schema:
    payload = {
        'values': ['question 1', 'question 2']
    }

    Output schema:
    result = {
        'predictions': [
            {
                'fields': ['answer', 'reference_documents'],
                'values': [
                    ['answer 1', [ {'page_content': 'page content 1',
                                    'metadata':     'metadata 1'} ]],
                    ['answer 2', [ {'page_content': 'page content 2',
                                    'metadata':     'metadata 2'} ]]
                ]
            }
        ]
    }
    """

    from ibm_watsonx_ai import APIClient, Credentials
    from ibm_watsonx_ai.foundation_models import ModelInference
    from ibm_watsonx_ai.foundation_models.extensions.rag import Retriever, VectorStore
    from ibm_watsonx_ai.foundation_models.extensions.rag.pattern.prompt_builder import (
        build_prompt,
    )
    from ibm_watsonx_ai.metanames import GenTextParamsMetaNames

    client = APIClient(
        credentials=Credentials(url=custom.get("url"), token=context.generate_token()),
        space_id=custom.get("space_id"),
    )
    vector_store = VectorStore.from_dict(client=client, data=custom["vector_store"])
    retriever = Retriever.from_vector_store(
        vector_store=vector_store, init_parameters=custom["retriever"]
    )
    prompt_template_text = custom["prompt_template_text"]
    context_template_text = custom["context_template_text"]
    model = ModelInference(api_client=client, **custom["model"])
    model_specs = client.foundation_models.get_model_specs(model_id=model.model_id)
    model_max_new_tokens = (model.params or {}).get(
        GenTextParamsMetaNames.MAX_NEW_TOKENS, 20
    )
    model_max_input_tokens = (
        model_specs["model_limits"]["max_sequence_length"] - model_max_new_tokens
    )

    def generate(context):
        client.set_token(context.get_token())
        payload = context.get_json()
        result = {"predictions": [{"fields": ["answer", "reference_documents"]}]}

        all_prompts = []
        all_retrieved_docs = []

        for question in payload["values"]:
            retrieved_docs = retriever.retrieve(query=question)
            all_retrieved_docs.append(retrieved_docs)
            reference_documents = [doc.page_content for doc in retrieved_docs]

            prompt_input_text = build_prompt(
                prompt_template_text=prompt_template_text,
                context_template_text=context_template_text,
                question=question,
                reference_documents=reference_documents,
                model_max_input_tokens=model_max_input_tokens,
            )
            all_prompts.append(prompt_input_text)

        answers = [model.generate_text(prompt=prompt) for prompt in all_prompts]

        predictions = [
            [
                answer,
                [
                    {"page_content": doc.page_content, "metadata": doc.metadata}
                    for doc in retrieved_docs
                ],
            ]
            for answer, retrieved_docs in zip(answers, all_retrieved_docs)
        ]

        result["predictions"][0]["values"] = predictions

        return {"body": result}

    return generate

### Test the function locally

To test our solution we can query the function locally without deploying.

In [20]:
questions_and_answers = {
    "what are the names of founding fathers of the united states?": "Thomas Jefferson::James Madison::John Jay::George Washington::John Adams::Benjamin Franklin::Alexander Hamilton",
    "who played in the super bowl in 2013?": "Baltimore Ravens::San Francisco 49ers",
    "when did bucharest become the capital of romania?": "1862",
}

Define a helper function for formatting the response:

In [21]:
def print_rag_response(response):
    for question, (answer, reference_docs) in zip(
        questions_and_answers.keys(), response["predictions"][0]["values"]
    ):
        verbose_search(question, [Document(**d) for d in reference_docs])
        display(Markdown(f"**Answer:** {answer}"))

Questions have to be provided in the payload that have format provided below.

In [22]:
payload = {"values": list(questions_and_answers.keys())}

In [23]:
from ibm_watsonx_ai.deployments import RuntimeContext

context = RuntimeContext(api_client=client)
generate_function = deployable_ai_service(context=context, **custom)

Unsetting the space_id ...
Unsetting the project_id ...


In [24]:
context.request_payload_json = payload
response = generate_function(context=context)
print_rag_response(response["body"])

**Question:** what are the names of founding fathers of the united states?

,page_content,id,pk,title
0,Founding Fathers of the United States,878.0,98aa4910657083171c7baf0a4f07b36ca1736a8fa509cb...,Founding Fathers of the United States
1,Founding Fathers of the United States,879.0,42d8e6fe20269c23a85346969087d5dd6ebf0e00314479...,Founding Fathers of the United States
2,Founding Fathers of the United States,880.0,e731e99f25e0338de640caab51397e36112000f5cecae2...,Founding Fathers of the United States
3,Founding Fathers of the United States,881.0,8c018ebca9d712a9ed9daa914c290de8f93c329831582e...,Founding Fathers of the United States
4,Founding Fathers of the United States,882.0,5f2ef7fbdd0a03d4c16a685060d0a6e6b91e44bca99670...,Founding Fathers of the United States


**Answer:**  The Founding Fathers of the United States include individuals such as George Washington, Thomas Jefferson, John Adams, James Madison, Benjamin Franklin, Alexander Hamilton, James Monroe, and other key figures who played a crucial role in shaping America's history. These individuals were instrumental in drafting the Declaration of Independence, the United States Constitution, and other foundational documents that have helped shape the country's governance and principles. Notable individuals include George Mason, Patrick Henry, and Roger Sherman, who also contributed significantly to the country's founding.

**Question:** who played in the super bowl in 2013?

,page_content,title,pk,id
0,"responded to the claim on Twitter in jest , tw...",Super Bowl XLVII,4035f1be2332109f293fd4520860b08339d2374206730d...,848.0
1,Super Bowl XLVII - wikipedia Super Bowl XLVII ...,Super Bowl XLVII,d6ac7acbbf3cfa1d40295c0f3bf426c5eeaf5a5b94e838...,818.0
2,Broadcast Schedule : NFL Super Bowl XLVII -- 2...,Super Bowl XLVII,4784977411c929284d9fbdebb97afce944fc3433a8de28...,863.0
3,: Super Bowl 2012 National Football League sea...,Super Bowl XLVII,1494ee13a6d4ad66c0691b619bce671b52454b2b363134...,876.0
4,Opponents Announced '' . NewOrleansSaints.com ...,Super Bowl XLVII,6a9d55d7d47ec5a64744d8b46f661784185fa25b29c5e0...,856.0


**Answer:**  Baltimore Ravens and San Francisco 49ers.  The game was played on February 3, 2013.  The Ravens won with a score of 34-31.  The game MVP was Joe Flacco, the quarterback for the Ravens.  The game was played at the Mercedes-Benz Superdome in New Orleans, Louisiana.  The game was refereed by Jerome Boger.  The game was broadcast on CBS by Jim Nantz and Phil Simms.  Alicia Keys sang the national anthem.  The coin toss was performed by Larry Allen.  The game was known as the "Harbaugh Bowl" because both teams were coached by members of the Harbaugh family.  The game was played in front of a crowd of 71,024.  The game was a rematch of the 2011 Super Bowl between the Ravens and the 49ers.  The Ravens were led by quarterback Joe Flacco, running back Ray Rice, and wide receiver Anquan Boldin

**Question:** when did bucharest become the capital of romania?

,page_content,id,pk,title
0,destroying a third of the city . Ottoman massa...,948.0,b28759a8b47ed9226b5e06b553deade254920c17a71193...,Bucharest
1,documents in 1459 . It became the capital of R...,944.0,e5bad390c50dd4d59fafaf1086dd10a4c2cb548ab91b11...,Bucharest
2,exist . Bucharest 's population experienced tw...,965.0,702af1d28cfa69c4fffcb934bbf2074228de67b5530364...,Bucharest
3,. I.C. Brătianu Boulevard in the 1930s Between...,948.0,81a7e73e848c22ab812a588572de42f837316639c57802...,Bucharest
4,Bucharest,942.0,9701f9e1b470084684aeb08a6187317587ccae416979ef...,Bucharest


**Answer:** 1862. after wallachia and moldavia were united to form the principality of romania. In 1881, it became the political centre of the newly proclaimed Kingdom of Romania under King Carol I. During the second half of the 19th century, the city's population increased dramatically, and a new period of urban development began. During this period, Bucharest became the capital of Romania. In 1862, it became the capital of Romania. In 1881, it became the political centre of the newly proclaimed Kingdom of Romania under King Carol I. During the second half of the 19th century, the city's population increased dramatically, and a new period of urban development began. In 1862, Bucharest became the capital of Romania. In 1881, it became the political centre of the newly proclaimed Kingdom of Romania under King Carol I. During the second half of the 19th century, the city's population increased dramatically, and a new period

### Deploy RAGPattern

Deployment is done by storing the defined RAG function and then by creating a deployed asset. It would be now accessed as an endpoint that we can score. Before the deployment a custom software specification need to be created to use langchain packages.

In [25]:
conda_yml = f"""
        name: python311
        channels:
          - empty
        dependencies:
          - pip:
            - ibm-watsonx-ai[rag]
        prefix: /opt/anaconda3/envs/python311
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(conda_yml)

In [26]:
base_sw_spec_id = client.software_specifications.get_id_by_name("runtime-24.1-py3.11")
meta_prop_pkg_extn = {
    client.package_extensions.ConfigurationMetaNames.NAME: "rag_sample_spec_24.1-py3.11",
    client.package_extensions.ConfigurationMetaNames.DESCRIPTION: "Environment with langchain",
    client.package_extensions.ConfigurationMetaNames.TYPE: "conda_yml",
}

pkg_extn_details = client.package_extensions.store(
    meta_props=meta_prop_pkg_extn, file_path="config.yaml"
)
pkg_extn_id = client.package_extensions.get_id(pkg_extn_details)
pkg_extn_id

Creating package extension
SUCCESS


'8b6513e8-26b6-4584-89d5-d0f89deeef7f'

In [27]:
meta_prop_sw_spec = {
    client.software_specifications.ConfigurationMetaNames.NAME: "RAG AI service watsonx.ai custom software specification",
    client.software_specifications.ConfigurationMetaNames.DESCRIPTION: "Software specification for AI service deployment",
    client.software_specifications.ConfigurationMetaNames.BASE_SOFTWARE_SPECIFICATION: {
        "guid": base_sw_spec_id
    },
}

sw_spec_details = client.software_specifications.store(meta_props=meta_prop_sw_spec)
sw_spec_id = client.software_specifications.get_id(sw_spec_details)
client.software_specifications.add_package_extension(sw_spec_id, pkg_extn_id)
sw_spec_id

SUCCESS


'b6e3ed1d-8ffb-4de4-835b-82d20b4438ae'

In [28]:
meta_props = {
    client.repository.AIServiceMetaNames.NAME: "RAG AI service SDK",
    client.repository.AIServiceMetaNames.SOFTWARE_SPEC_ID: sw_spec_id,
}
stored_ai_service_details = client.repository.store_ai_service(
    deployable_ai_service, meta_props
)

In [29]:
ai_service_id = client.repository.get_ai_service_id(stored_ai_service_details)
ai_service_id

'9f8812b7-036a-46f0-99df-dcd866460113'

In [30]:
meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: "RAG pattern AI service",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
    client.deployments.ConfigurationMetaNames.CUSTOM: custom,
}

deployment_details = client.deployments.create(ai_service_id, meta_props)



######################################################################################

Synchronous deployment creation for id: '9f8812b7-036a-46f0-99df-dcd866460113' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
............
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='ab0ee6dd-1c95-436f-8a9e-6c13f9a09b57'
-----------------------------------------------------------------------------------------------




### Test the deployed function

RAG service is now deployed on our space. To test our solution we can run the cell below. Questions have to be provided in the payload that have format provided below.

In [31]:
deployment_id = client.deployments.get_id(deployment_details)

In [32]:
response = client.deployments.run_ai_service(deployment_id, payload)
print_rag_response(response)

**Question:** what are the names of founding fathers of the united states?

,page_content,title,pk,id
0,Founding Fathers of the United States,Founding Fathers of the United States,98aa4910657083171c7baf0a4f07b36ca1736a8fa509cb...,878.0
1,Founding Fathers of the United States,Founding Fathers of the United States,42d8e6fe20269c23a85346969087d5dd6ebf0e00314479...,879.0
2,Founding Fathers of the United States,Founding Fathers of the United States,e731e99f25e0338de640caab51397e36112000f5cecae2...,880.0
3,Founding Fathers of the United States,Founding Fathers of the United States,8c018ebca9d712a9ed9daa914c290de8f93c329831582e...,881.0
4,Founding Fathers of the United States,Founding Fathers of the United States,5f2ef7fbdd0a03d4c16a685060d0a6e6b91e44bca99670...,882.0


**Answer:**  The Founding Fathers of the United States were a group of individuals who played a significant role in the formation of the country. Some of the most notable Founding Fathers include George Washington, Thomas Jefferson, John Adams, James Madison, Benjamin Franklin, Alexander Hamilton, and many others who contributed to the drafting and signing of the Declaration of Independence and the United States Constitution. These individuals were instrumental in shaping the country's government, politics, and values. 
The Founding Fathers of the United States were a group of individuals who played a significant role in the formation of the country. Some of the most notable Founding Fathers include George Washington, Thomas Jefferson, John Adams, James Madison, Benjamin Franklin, Alexander Hamilton, and many others who contributed to the drafting and signing of the Declaration of Independence and the United States Constitution. These individuals were instrumental in shaping the country's government, politics, and values. 
The Founding Fathers of the United States were a group of individuals who played a significant role in the

**Question:** who played in the super bowl in 2013?

,page_content,title,pk,id
0,"responded to the claim on Twitter in jest , tw...",Super Bowl XLVII,4035f1be2332109f293fd4520860b08339d2374206730d...,848.0
1,Super Bowl XLVII - wikipedia Super Bowl XLVII ...,Super Bowl XLVII,d6ac7acbbf3cfa1d40295c0f3bf426c5eeaf5a5b94e838...,818.0
2,Broadcast Schedule : NFL Super Bowl XLVII -- 2...,Super Bowl XLVII,4784977411c929284d9fbdebb97afce944fc3433a8de28...,863.0
3,: Super Bowl 2012 National Football League sea...,Super Bowl XLVII,1494ee13a6d4ad66c0691b619bce671b52454b2b363134...,876.0
4,Opponents Announced '' . NewOrleansSaints.com ...,Super Bowl XLVII,6a9d55d7d47ec5a64744d8b46f661784185fa25b29c5e0...,856.0


**Answer:**  Baltimore Ravens and San Francisco 49ers.  The game was played at the Mercedes-Benz Superdome in New Orleans, with the Ravens winning 34-31.  The game was played on February 3, 2013.  The MVP was Joe Flacco.   The 49ers were favored by 4 points.  The game was refereed by Jerome Boger.  The game was attended by 71,024 people.   The national anthem was sung by Alicia Keys.  The coin toss was performed by Larry Allen.  The game was broadcast by CBS, with Jim Nantz and Phil Simms as the announcers.  The game was the 47th Super Bowl, also known as Super Bowl XLVII.   The game was also known as "The Harbaugh Bowl" because the two coaches, John Harbaugh of the Ravens and Jim Harbaugh of the 49ers, are brothers.  The game was played indoors because it was held in

**Question:** when did bucharest become the capital of romania?

,page_content,title,pk,id
0,destroying a third of the city . Ottoman massa...,Bucharest,b28759a8b47ed9226b5e06b553deade254920c17a71193...,948.0
1,documents in 1459 . It became the capital of R...,Bucharest,e5bad390c50dd4d59fafaf1086dd10a4c2cb548ab91b11...,944.0
2,exist . Bucharest 's population experienced tw...,Bucharest,702af1d28cfa69c4fffcb934bbf2074228de67b5530364...,965.0
3,. I.C. Brătianu Boulevard in the 1930s Between...,Bucharest,81a7e73e848c22ab812a588572de42f837316639c57802...,948.0
4,Bucharest,Bucharest,9701f9e1b470084684aeb08a6187317587ccae416979ef...,942.0


**Answer:** 1862.

<a id="evaluate"></a>
## Calculate rougeL metric 
Calculate rougeL recall score to verify expected answer presence in generated response.

In [33]:
text_responses = [v[0] for v in response["predictions"][0]["values"]]
targets = [answer for answer in questions_and_answers.values()]

In [34]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
scores = [
    scorer.score(target, prediction)
    for target, prediction in zip(targets, text_responses)
]
mean_rougeL = sum([s["rougeL"].recall for s in scores]) / len(questions_and_answers)

print(f"Mean rougeL recall score: {mean_rougeL}")

Mean rougeL recall score: 0.9523809523809524


<a id="summary"></a>
## Summary and next steps

You successfully completed this notebook!

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors:
**Dominik Zimny**, Software Engineer at watsonx.ai

**Mateusz Szewczyk**, Software Engineer at watsonx.ai

Copyright © 2024-2026 IBM. This notebook and its source code are released under the terms of the MIT License.